# 14. File I/O & Serialization: Beginner Guide

### 🌟 What is File I/O & Binary Serialization in NumPy?
NumPy provides specialized binary formats (`.npy` for single arrays, `.npz` for compressed multi-array archives) that load and save numerical data orders of magnitude faster than CSV files, as well as memory-mapped files (`np.memmap`) for datasets larger than RAM.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Binary Serialization (`.npy`)**: Covers `np.save()` and `np.load()`.
- **Multi-Array Archives (`.npz`)**: Covers `np.savez()` and `np.savez_compressed()`.
- **Memory-Mapped Files (`np.memmap`)**: Out-of-core virtual memory mapping for arrays larger than RAM.
- **Text Parsing**: Covers `np.loadtxt()` and `np.genfromtxt()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 Binary Persistence: `np.save()` & `np.load()`
Serializes transaction amounts to fast `.npy` binary format. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.save('scratch/tx_amounts.npy', amounts)`


In [2]:
os.makedirs('scratch', exist_ok=True)
np.save('scratch/tx_amounts.npy', amounts)
loaded_amounts = np.load('scratch/tx_amounts.npy')
print('Saved & Loaded array equal?:', np.array_equal(amounts, loaded_amounts))

Saved & Loaded array equal?: True


### 🔹 Uncompressed Archive: `np.savez()`
Bundles amounts, fraud flags, and account ages into a single `.npz` archive. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.savez('scratch/transactions_all.npz', amounts=amounts, fraud=fraud_flags)`


In [3]:
np.savez('scratch/transactions_all.npz', amounts=amounts, fraud=fraud_flags, ages=account_ages)
with np.load('scratch/transactions_all.npz') as arch:
    print('Archived Keys:', arch.files)
    print('Loaded fraud array shape:', arch['fraud'].shape)

Archived Keys: ['amounts', 'fraud', 'ages']
Loaded fraud array shape: (14262,)


### 🔹 Compressed Multi-Array Archive: `np.savez_compressed()`
Applies zip compression saving 70% disk space. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.savez_compressed('scratch/tx_compressed.npz', amounts=amounts, fraud=fraud_flags)`


In [4]:
np.savez_compressed('scratch/tx_compressed.npz', amounts=amounts, fraud=fraud_flags)
print('Compressed npz written. Size:', os.path.getsize('scratch/tx_compressed.npz'), 'bytes')

Compressed npz written. Size: 48371 bytes


### 🔹 Out-of-Core Memory Mapping: `np.memmap()`
Memory-maps binary transaction records directly into virtual RAM without loading file into heap. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** `.apply()` runs a standard Python loop row-by-row. Whenever possible, use built-in vectorized operations (`df['a'] + df['b']`) which run up to 100x faster!.

**Syntax:** `np.memmap('scratch/large_tx.dat', dtype='float64', mode='w+', shape=amounts.shape)`


In [5]:
mmap_tx = np.memmap('scratch/mmap_transactions.dat', dtype='float64', mode='w+', shape=amounts.shape)
mmap_tx[:] = amounts[:]
mmap_tx.flush()
print('Memory-mapped transaction buffer created. Shape:', mmap_tx.shape)

Memory-mapped transaction buffer created. Shape: (14262,)


### 🔹 Text Parsing: `np.loadtxt()` & `np.genfromtxt()`
Parses comma-delimited numeric columns directly from CSV. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.genfromtxt('data/raw_transactions.csv', delimiter=',', skip_header=1, usecols=(3, 7))`


In [6]:
tx_numeric_cols = np.genfromtxt(csv_path, delimiter=',', skip_header=1, usecols=(3, 7), max_rows=5)
print('Parsed Numeric Columns (amount, account_age):\n', tx_numeric_cols)

Parsed Numeric Columns (amount, account_age):
 [[1216.33   56.  ]
 [ 324.99  112.  ]
 [ 136.66   68.  ]
 [ 124.21   50.  ]
 [1284.68   96.  ]]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Streaming Chunked Aggregation on Memmap

**Approach:** Calculate total transaction spend across memory-mapped file in chunks.
**Syntax:** `mmap_tx[start:end].sum()`


In [7]:
chunk_sz = 2500
total_mmap_sum = sum(mmap_tx[i:i+chunk_sz].sum() for i in range(0, len(mmap_tx), chunk_sz))
print(f'Total Spend across Memmap Chunks: ${total_mmap_sum:,.2f}')

Total Spend across Memmap Chunks: $14,349,538.14
